### DataStax VectorDB

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

ASTRA_DB_API_ENDPOINT = os.getenv("ASTRA_DB_API_ENDPOINT")
ASTRA_DB_APPLICATION_TOKEN = os.getenv("ASTRA_DB_APPLICATION_TOKEN")

In [4]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name ="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
embeddings

HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [6]:
from langchain_astradb import AstraDBVectorStore
vector_store = AstraDBVectorStore(
    embedding=embeddings,
    api_endpoint=ASTRA_DB_API_ENDPOINT,
    collection_name="astra_vector_langchain",
    token= ASTRA_DB_APPLICATION_TOKEN,
    namespace=None,
    
)

vector_store

In [7]:
from langchain_core.documents import Document

document_1 = Document(
    page_content="I had chocolate chip pancakes and scrambled eggs for breakfast this morning.",
    metadata={"source": "tweet"},
)

document_2 = Document(
    page_content="The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.",
    metadata={"source": "news"},
)

document_3 = Document(
    page_content="Building an exciting new project with LangChain - come check it out!",
    metadata={"source": "tweet"},
)

document_4 = Document(
    page_content="Robbers broke into the city bank and stole $1 million in cash.",
    metadata={"source": "news"},
)

document_5 = Document(
    page_content="Wow! That was an amazing movie. I can't wait to see it again.",
    metadata={"source": "tweet"},
)

document_6 = Document(
    page_content="Is the new iPhone worth the price? Read this review to find out.",
    metadata={"source": "website"},
)

document_7 = Document(
    page_content="The top 10 soccer players in the world right now.",
    metadata={"source": "website"},
)

document_8 = Document(
    page_content="LangGraph is the best framework for building stateful, agentic applications!",
    metadata={"source": "tweet"},
)

document_9 = Document(
    page_content="The stock market is down 500 points today due to fears of a recession.",
    metadata={"source": "news"},
)

document_10 = Document(
    page_content="I have a bad feeling I am going to get deleted :(",
    metadata={"source": "tweet"},
)

documents = [
    document_1,
    document_2,
    document_3,
    document_4,
    document_5,
    document_6,
    document_7,
    document_8,
    document_9,
    document_10,
]

documents

[Document(metadata={'source': 'tweet'}, page_content='I had chocolate chip pancakes and scrambled eggs for breakfast this morning.'),
 Document(metadata={'source': 'news'}, page_content='The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.'),
 Document(metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(metadata={'source': 'news'}, page_content='Robbers broke into the city bank and stole $1 million in cash.'),
 Document(metadata={'source': 'tweet'}, page_content="Wow! That was an amazing movie. I can't wait to see it again."),
 Document(metadata={'source': 'website'}, page_content='Is the new iPhone worth the price? Read this review to find out.'),
 Document(metadata={'source': 'website'}, page_content='The top 10 soccer players in the world right now.'),
 Document(metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic application

In [8]:
vector_store.add_documents(documents=documents)

['86bcd777fb734178b032806993fbd811',
 '075109d6b13c4c97a8111454b0a9f472',
 'c34b54c95c1b49a3940bd03103658abe',
 'fc6d39fb801a4bc68eb62d80b8fd201a',
 '38665b312508404b97b755ae271e5889',
 '79e04ed62d144d259f7f5618c5451751',
 '0e990f00a91942e1b82631d63ca4fd04',
 '09faaf1fc91b41eebdb9e8d63676a087',
 '0d4210351bd8430fb299eb4da3fffdb2',
 '3ba0346fb7b949778cc4f5b74323ed67']

### Search from Vector Store DB

In [10]:
vector_store.similarity_search("What is the weather")

[Document(id='075109d6b13c4c97a8111454b0a9f472', metadata={'source': 'news'}, page_content='The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.'),
 Document(id='0d4210351bd8430fb299eb4da3fffdb2', metadata={'source': 'news'}, page_content='The stock market is down 500 points today due to fears of a recession.'),
 Document(id='09faaf1fc91b41eebdb9e8d63676a087', metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic applications!'),
 Document(id='0e990f00a91942e1b82631d63ca4fd04', metadata={'source': 'website'}, page_content='The top 10 soccer players in the world right now.')]

In [11]:
result = vector_store.similarity_search(
    "Langchain provide abstractions to make working with LLMs easy",
    k=3,
    filter = {"source":"tweet"}
)

for res in result:
    print(f'* "{res.page_content}", metadata={res.metadata}')

* "Building an exciting new project with LangChain - come check it out!", metadata={'source': 'tweet'}
* "LangGraph is the best framework for building stateful, agentic applications!", metadata={'source': 'tweet'}
* "I have a bad feeling I am going to get deleted :(", metadata={'source': 'tweet'}


In [12]:
result = vector_store.similarity_search_with_score(
    "Langchain provide abstractions to make working with LLMs easy",
    k=3,
    filter = {"source":"tweet"}
)

for res, score in result:
    print(f'* [SIM={score:.2f}] "{res.page_content}", metadata={res.metadata}')

* [SIM=0.79] "Building an exciting new project with LangChain - come check it out!", metadata={'source': 'tweet'}
* [SIM=0.73] "LangGraph is the best framework for building stateful, agentic applications!", metadata={'source': 'tweet'}
* [SIM=0.51] "I have a bad feeling I am going to get deleted :(", metadata={'source': 'tweet'}


In [18]:
### Retriever
retriever = vector_store.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={
        "k": 1,
        "score_threshold": 0.5
    }
)
retriever.invoke("Stealing from the bank is crime", filter={"source": "news"})

[Document(id='fc6d39fb801a4bc68eb62d80b8fd201a', metadata={'source': 'news'}, page_content='Robbers broke into the city bank and stole $1 million in cash.')]

In [19]:
### Retriever
retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 1
    },
        
)
retriever.invoke("Stealing from the bank is crime", filter={"source": "news"})

[Document(id='fc6d39fb801a4bc68eb62d80b8fd201a', metadata={'source': 'news'}, page_content='Robbers broke into the city bank and stole $1 million in cash.')]